In [ ]:
import subprocess
import time
import threading
import os

print("Installing zstd...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# Function to run Ollama server
def run_ollama_server():
    subprocess.run(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Starting Ollama server...")
server_thread = threading.Thread(target=run_ollama_server, daemon=True)
server_thread.start()

time.sleep(10)

print("Pulling llama3.1:8b (this takes 5-10 minutes)...")
subprocess.run(["ollama", "pull", "llama3.1:8b"], check=True)

print("Ollama ready!")

In [ ]:
import os, sys, json, time, random, re
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_FILES = {
    "dev_track_a":  "dev_track_a.jsonl",
    "test_track_a": "test_track_a.jsonl",
    "test_track_b": "test_track_b.jsonl",
}

CACHE_PATH   = "/kaggle/working/aspects_cache_llm_extracted_kaggle.json"
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_URL   = "http://localhost:11434/api/generate"

# COMBINED PROMPT
# Single prompt extracting all three aspects in one LLM call.
# Returns a JSON object with keys "coa", "outcomes", "theme".
#
# Design principles:
#   - All three aspect definitions in one context so the model sees
#     how they DIFFER from each other (critical for correct abstraction)
#   - One shared example showing all three aspects for the SAME story
#     (contrast between aspects is clearer than three separate examples)
#   - JSON output enforced via Ollama format parameter (valid JSON guaranteed)
#   - Explicit "having..." prohibition stops outcomes over-generation
#   - Role label requirement for CoA + Theme prevents name leakage
#   - Tight length guidance in output format section

# System message — role and behavioural constraints
SYSTEM_MSG = (
    "You are a precise narrative analyst specialising in story structure. "
    "You follow instructions exactly. "
    "You always output valid JSON with no markdown fences, no extra keys, "
    "and no text outside the JSON object."
)

# Combined extraction prompt — one call, three aspects
COMBINED_PROMPT = """Analyse the story summary and extract three narrative aspects.
Return ONLY valid JSON with keys: "coa", "outcomes", "theme".
No extra text.

━━━ CORE PRINCIPLE ━━━
Each aspect must capture DIFFERENT information:
- COA = process (what happens)
- OUTCOMES = final state (what is true at the end)
- THEME = abstract meaning (what it represents)

Avoid overlap between them.

━━━━━━━━━━━━━━━━━━━━━━━
COA (Course of Action)
━━━━━━━━━━━━━━━━━━━━━━━
Describe the FULL causal sequence of events.

REQUIREMENTS:
- 3-6 numbered steps (1. 2. 3. ...)
- MUST include the FINAL transition into resolution
- Each step = one causal event
- Use abstract action types (escape, betrayal, investigation, confrontation, sacrifice)

USE:
- role labels (protagonist, antagonist, authority, ally)
- generic locations (city, prison, battlefield)

FORBIDDEN:
- character names
- specific places
- themes or emotions
- vague verbs ("deals with", "goes through")

GOOD:
"protagonist investigates → discovers threat → confronts antagonist → resolves conflict"

━━━━━━━━━━━━━━━━━━━━━━━
OUTCOMES (STRICT FORMAT)
━━━━━━━━━━━━━━━━━━━━━━━
Describe ONLY the final stable state.

Write EXACTLY 2 sentences:

Sentence 1:
- protagonist final status (success / failure / survival / transformation)

Sentence 2:
- type of resolution:
  - conflict_resolved / unresolved / partial
  - + nature of change (personal / relational / systemic)

FORBIDDEN:
- "having..." clauses
- process descriptions
- vague words like "things improve"

GOOD:
"Protagonist survives and achieves escape from immediate danger. The conflict is partially resolved with personal transformation but broader threats remain."

━━━━━━━━━━━━━━━━━━━━━━━
THEME (NORMALIZED)
━━━━━━━━━━━━━━━━━━━━━━━
Write 2-4 SHORT phrases (not full sentences).

Each phrase must be:
- abstract
- generalizable across stories

FORMAT:
"theme1; theme2; theme3"

GOOD:
"survival under adversity; loyalty vs duty; human resilience"

BAD:
"The story explores how a man struggles..."

━━━━━━━━━━━━━━━━━━━━━━━
EXAMPLE
━━━━━━━━━━━━━━━━━━━━━━━

Story:
"A young man injures his brother, is placed under supervision, falsely accused, and later proven innocent."

Output:
{{
  "coa": "1. Protagonist commits violence and is processed by authority.\n2. Authority imposes supervision and assigns a helper.\n3. Community falsely accuses protagonist, escalating conflict.\n4. Evidence emerges that clears protagonist and resolves accusations.",
  "outcomes": "Protagonist is exonerated and transitions to a stable path of self-improvement. The conflict is resolved with personal transformation and social reintegration.",
  "theme": "redemption; social stigma; justice vs prejudice"
}}

━━━━━━━━━━━━━━━━━━━━━━━
NOW ANALYSE
━━━━━━━━━━━━━━━━━━━━━━━

Story: {story}

Output:
"""

# Per-aspect validation thresholds
ASPECT_VALIDATORS = {
    "coa": {
        "min_len": 80,
        "max_len": 700,
        "warn_if_absent": r"^\s*1[\.\)]",
        "warn_msg": "CoA should start with numbered step '1.' or '1)'"
    },
    "outcomes": {
        "min_len": 40,
        "max_len": 300,
        "warn_if_absent": None,
        "warn_msg": ""
    },
    "theme": {
        "min_len": 60,
        "max_len": 450,
        "warn_if_absent": None,
        "warn_msg": ""
    },
}

# STORY COLLECTION

def _norm_key(text):
    return " ".join(str(text).split())


def collect_stories(data_files):
    seen, stories = set(), []

    def _add(text, title, source):
        if text is None:
            return
        key = _norm_key(text)
        if key and key not in seen and len(key) >= 20:
            seen.add(key)
            stories.append({"text": key, "title": title, "source": source})

    for name, path in data_files.items():
        if not Path(path).exists():
            print(f"  [skip] {path} not found")
            continue
        print(f"  Reading {path} ...")
        with open(path, encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                if name == "test_track_b":
                    _add(obj.get("text", ""),
                         obj.get("story_details", {}).get("title", ""), name)
                else:
                    for field, det_key in [
                        ("anchor_text", "story_details_anchor"),
                        ("text_a",      "story_details_a"),
                        ("text_b",      "story_details_b"),
                    ]:
                        _add(obj.get(field, ""),
                             obj.get(det_key, {}).get("title", ""), name)

    print(f"\nUnique stories collected: {len(stories)}")
    return stories


# CACHE

def load_cache(path):
    if Path(path).exists():
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        cache = {_norm_key(k): v for k, v in raw.items()}
        print(f"Cache loaded: {len(cache)} entries from {path}")
        return cache
    return {}


def save_cache(cache, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)


def is_complete(entry):
    return (isinstance(entry, dict)
            and all(entry.get(k, "").strip() for k in ("coa", "outcomes", "theme")))



# RESPONSE CLEANING & VALIDATION

# Strip preamble echoes from LLM responses (defence-in-depth after JSON parsing)
_PREAMBLE_RE = re.compile(
    r'^(?:'
    # "here is/are [the/a] <any phrase>" followed by newline
    r'here (?:is|are)(?: the| a)?[^\n:]{0,80}?[\s:.-]*\n+'
    r'|'
    # "Certainly!/Sure!" acknowledgements
    r'(?:certainly|sure|of course|absolutely)[!,.]?[^\n]*\n*'
    r'|'
    # Explicit aspect label ("Course of Action:", "Outcomes:", etc.)
    r'(?:course of action|outcomes?|abstract theme|response|answer)\s*[:：]\s*\n*'
    r')',
    re.IGNORECASE
)


def _clean_response(text):
    """Remove preamble echoes and leading labels from LLM responses."""
    if not text:
        return ""
    text = text.strip()
    for _ in range(4):
        cleaned = _PREAMBLE_RE.sub("", text).strip()
        if cleaned == text:
            break
        text = cleaned
    text = re.sub(r"^\s*\n+", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _validate_aspect(aspect_name, text, story_preview):
    """Print a warning if an extracted aspect looks malformed."""
    v = ASPECT_VALIDATORS.get(aspect_name, {})
    if not v:
        return
    issues = []
    if len(text) < v.get("min_len", 0):
        issues.append(f"too short ({len(text)} chars, min={v['min_len']})")
    if len(text) > v.get("max_len", 9999):
        issues.append(f"very long ({len(text)} chars) — may be over-generated")
    pat = v.get("warn_if_absent")
    if pat and not re.search(pat, text, re.MULTILINE):
        issues.append(v.get("warn_msg", ""))
    if aspect_name in ("coa", "theme") and re.search(r'\([A-Z][a-z]+ [A-Z][a-z]+\)', text):
        issues.append("may contain actor name in parentheses")
    if issues:
        print(f"    \u26a0  {aspect_name}: {'; '.join(issues)}")
        print(f"       story: {story_preview[:60]}...")
        print(f"       text:  {text[:100]}")

# OLLAMA BACKEND

def _ollama_available():
    try:
        import urllib.request
        urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", ""), timeout=2)
        return True
    except Exception:
        return False


def check_ollama_gpu():
    import urllib.request
    try:
        with urllib.request.urlopen(
            OLLAMA_URL.replace("/api/generate", "/api/ps"), timeout=5
        ) as resp:
            ps = json.loads(resp.read())
        for m in ps.get("models", []):
            vram = m.get("size_vram", 0)
            status = "GPU ✓" if vram > 0 else "CPU ⚠ slow!"
            print(f"  Ollama: {m.get('name')}  VRAM={vram/1e9:.1f}GB  {status}")
    except Exception:
        pass


def _ollama_generate(prompt, max_tokens=500, json_mode=False):
    """
    Call Ollama API.
    json_mode=True adds format="json" to the request — Ollama guarantees
    the response is valid JSON (llama3.1:8b supports this natively).
    """
    import urllib.request
    payload_dict = {
        "model":   OLLAMA_MODEL,
        "prompt":  prompt,
        "stream":  False,
        "options": {
            "num_predict":    max_tokens,
            "temperature":    0.1,   # low — factual extraction
            "top_p":          0.9,
            "repeat_penalty": 1.1,   # discourage prompt echo
        },
    }
    if json_mode:
        payload_dict["format"] = "json"

    payload = json.dumps(payload_dict).encode()
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=180) as resp:
        return json.loads(resp.read()).get("response", "").strip()
def _parse_json_response(raw):
    """
    Parse JSON from LLM response. Handles:
      - Clean JSON objects
      - JSON wrapped in markdown fences (```json ... ```)
      - Partial JSON with recoverable structure
    Returns dict with keys "coa", "outcomes", "theme", or raises ValueError.
    """
    text = raw.strip()

    # Strip markdown fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    text = text.strip()

    # Attempt direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, dict) and all(k in obj for k in ("coa", "outcomes", "theme")):
            return obj
    except json.JSONDecodeError:
        pass

    # Recovery: extract individual fields with regex if JSON is malformed
    # (handles cases where the model emits valid field values but broken JSON syntax)
    recovered = {}
    for key in ("coa", "outcomes", "theme"):
        # Match: "key": "value" — handles escaped newlines and quotes
        pattern = rf'"{key}"\s*:\s*"((?:[^"\\]|\\.)*)"'
        m = re.search(pattern, text, re.DOTALL)
        if m:
            # Unescape the captured value
            recovered[key] = m.group(1).replace("\\n", "\n").replace('\\"', '"')

    if len(recovered) == 3:
        return recovered

    raise ValueError(f"Could not parse JSON response. Raw: {raw[:200]}")


def extract_with_ollama(story_text, title=""):
    """
    Extract all three narrative aspects in a single LLM call.
    Returns dict with keys: title, coa, outcomes, theme.
    """
    prompt   = SYSTEM_MSG + "\n\n" + COMBINED_PROMPT.format(story=story_text)
    aspects  = {"title": title}

    try:
        raw     = _ollama_generate(prompt, max_tokens=500, json_mode=True)
        parsed  = _parse_json_response(raw)
        aspects["coa"]      = _clean_response(parsed.get("coa",      ""))
        aspects["outcomes"] = _clean_response(parsed.get("outcomes", ""))
        aspects["theme"]    = _clean_response(parsed.get("theme",    ""))
    except Exception as e:
        print(f"    [ollama error] {e}")
        aspects["coa"] = aspects["outcomes"] = aspects["theme"] = ""
        return aspects

    # Validate each extracted aspect
    for asp in ("coa", "outcomes", "theme"):
        _validate_aspect(asp, aspects[asp], story_text)

    return aspects

# CACHE REPAIR

def repair_cache(cache_path):
    """
    Re-run _clean_response on all entries.
    Also flag entries where CoA is missing numbered steps (likely old-format).
    """
    cache = load_cache(cache_path)
    if not cache:
        print("Cache empty.")
        return
    cleaned_fields = 0
    suspicious = []
    for text, entry in cache.items():
        for asp in ("coa", "outcomes", "theme"):
            original = entry.get(asp, "")
            cleaned  = _clean_response(original)
            if cleaned != original:
                entry[asp] = cleaned
                cleaned_fields += 1
        # Flag CoA entries missing numbered structure (old extractions)
        coa = entry.get("coa", "")
        if coa and not re.search(r"^\s*1[\.\)]", coa, re.MULTILINE):
            suspicious.append((text[:60], coa[:80]))

    save_cache(cache, cache_path)
    print(f"Repair complete: {cleaned_fields} fields cleaned, "
          f"{len(suspicious)} CoA entries lack numbered steps.")
    if suspicious:
        print("First 3 suspicious CoA entries:")
        for story, coa in suspicious[:3]:
            print(f"  story: {story}...")
            print(f"  coa:   {coa}...")
    return cache


# QUALITY CHECK

def quality_check(cache, n=10):
    print("\n" + "=" * 70)
    print(f"  QUALITY CHECK — {min(n, len(cache))} random samples")
    print("=" * 70)

    for i, key in enumerate(random.sample(list(cache.keys()),
                                           min(n, len(cache))), 1):
        entry = cache[key]
        print(f"\n[{i}] {entry.get('title','(no title)')}")
        print(f"  Story : {key[:120]}...")
        print(f"  CoA   : {entry.get('coa','(empty)')[:200]}")
        print(f"  Out   : {entry.get('outcomes','(empty)')[:150]}")
        print(f"  Theme : {entry.get('theme','(empty)')[:150]}")
        for asp in ("coa", "outcomes", "theme"):
            _validate_aspect(asp, entry.get(asp, ""), key)

    total    = len(cache)
    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"\n  Total: {total}  Complete: {complete} ({complete/total*100:.1f}%)"
          if total else "\n  Cache empty.")

    # Check CoA structural quality: fraction with numbered steps
    coa_structured = sum(
        1 for v in cache.values()
        if re.search(r"^\s*1[\.\)]", v.get("coa", ""), re.MULTILINE)
    )
    print(f"  CoA with numbered steps: {coa_structured}/{total} "
          f"({coa_structured/total*100:.1f}%)" if total else "")
    print("=" * 70 + "\n")


# EXTRACTION LOOP

def run_extraction(stories, cache, dry_run=False):
    todo = [s for s in stories if not is_complete(cache.get(s["text"]))]
    if dry_run:
        todo = todo[:5]
        print(f"DRY RUN: processing {len(todo)} stories only\n")
    else:
        print(f"To process: {len(todo)}  "
              f"Already cached: {len(stories) - len(todo)}\n")

    if not todo:
        print("Nothing to do — all stories cached.")
        return cache

    start = time.time()
    errors = 0

    for i, story in enumerate(tqdm(todo, desc="Extracting aspects")):
        try:
            aspects = extract_with_ollama(story["text"], story["title"])
            cache[story["text"]] = aspects
        except Exception as e:
            print(f"\n  [ERROR] story {i}: {e}")
            cache[story["text"]] = {
                "title": story["title"], "coa": "", "outcomes": "", "theme": ""}
            errors += 1

        # Save every 10 stories — crash recovery
        if (i + 1) % 10 == 0 or i == len(todo) - 1:
            save_cache(cache, CACHE_PATH)

        # Progress ETA every 50 stories
        if (i + 1) % 50 == 0:
            elapsed   = time.time() - start
            rate      = (i + 1) / elapsed
            remaining = (len(todo) - i - 1) / rate / 60
            print(f"\n  [{i+1}/{len(todo)}] "
                  f"{rate:.2f} stories/s  ETA {remaining:.1f} min  "
                  f"errors: {errors}")

    elapsed = time.time() - start
    print(f"\nDone in {elapsed/60:.1f} min  |  errors: {errors}")
    return cache


# CONFIG & MAIN

class _Cfg:
    dry_run       = False   # True = process 5 stories only
    quality_check = False   # True = inspect cache and exit
    repair        = False   # True = clean existing cache entries and exit
    cache         = CACHE_PATH
    data_dir      = "/kaggle/input/similarity"      # empty = use DATA_FILES paths as-is


def main():
    args = _Cfg()

    global CACHE_PATH
    CACHE_PATH = args.cache

    data_files = (
        {name: str(Path(args.data_dir) / fname)
         for name, fname in DATA_FILES.items()}
        if args.data_dir else dict(DATA_FILES)
    )

    print("=" * 60)
    print("  Narrative Aspect Extraction (LLM)")
    print("=" * 60)
    print(f"  Cache   : {CACHE_PATH}")
    print(f"  Backend : Ollama ({OLLAMA_MODEL})")
    print()

    cache = load_cache(CACHE_PATH)

    if args.repair:
        repair_cache(CACHE_PATH)
        return

    if args.quality_check:
        quality_check(cache) if cache else print("Cache empty.")
        return

    if not _ollama_available():
        print("ERROR: Ollama not running. Start with: ollama serve")
        print("       Then pull model: ollama pull llama3.1:8b")
        sys.exit(1)

    check_ollama_gpu()

    print("Collecting stories ...")
    stories = collect_stories(data_files)
    print()

    cache = run_extraction(stories, cache, dry_run=args.dry_run)

    if not args.dry_run:
        quality_check(cache, n=5)

    complete = sum(1 for v in cache.values() if is_complete(v))
    print(f"Final: {len(cache)} entries, {complete} complete")
    print(f"Cache: {CACHE_PATH}")


if __name__ == "__main__":
    main()
